# 07. От DiT к CrossDiT: как denoiser устроен изнутри

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** проследить путь latent→patches→DiT blocks→output и отличить time-modulation от text cross-attention

**Перед началом:** K6 05–06; Lecture 4 / Lab 3. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/06_dit_and_flow_matching.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.2](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)
- [`core/tensors.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/tensors.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','kandinsky-6-components'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '07_dit_and_text_crossdit'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Что означает DiT

В [оригинальной статье DiT — Peebles & Xie](https://arxiv.org/abs/2212.09748) transformer обрабатывает
patches latent image вместо U-Net. Ключевой механизм conditioning — adaLN/adaLN-Zero.
Transformer architecture и training target — разные решения: image diffusion может предсказывать noise,
Kandinsky FM — velocity, distilled K6 — grid policy. Смена target не отменяет transformer backbone.

Сначала рассмотрим блок с self-attention и FFN. Затем добавим отдельный источник K/V — text.
Именно это различие делает архитектурное чтение понятнее, чем одно общее слово «Transformer».

[Технический отчёт K5](https://arxiv.org/abs/2511.14993), §architecture, связывает этот шаг с CrossDiT
Kandinsky. Термин здесь не означает image-classification CrossViT с patches двух размеров.


In [ ]:
D=native();tokens=torch.randn(1,8,48)
time_encoder=D.TimeEmbeddings(48,16);t=time_encoder(torch.tensor([500.]))
self_ff=D.TransformerEncoderBlock(48,16,96,8,engine='sdpa')
rope=native('core.components.rope').RoPE1D(8)(torch.arange(8))
with torch.no_grad():identity=self_ff(tokens,t,rope)
assert torch.equal(identity,tokens)
activate_gates(self_ff)
with torch.no_grad():changed=self_ff(tokens,t,rope)
metrics['self_ff_zero_gate_identity_error']=(identity-tokens).abs().max().item()
metrics['self_ff_active_change']=(changed-tokens).abs().mean().item()
print('Same native self-attention/FFN primitives; random weights:',metrics)


## 2. CrossDiT K5/K6: visual query, text K/V

Текст сначала проходит собственный refiner. Visual decoder block делает три residual updates:
self-attention, cross-attention к тексту, FFN. На каждую ветвь нужны shift/scale/gate — всего 9 векторов.
Text refiner использует две ветви и 6 векторов. Time+pooled conditioning управляет modulation,
а token-level Qwen context поступает в K/V cross-attention. Эти каналы нельзя подменять друг другом.

Ниже **настоящий native video-only CrossDiT**, с уменьшенными размерами и случайными весами.
Это T2V-ветвь того же класса, не претензия на полную реализацию оригинального ImageNet DiT.


In [ ]:
from labkit.torch_reference import tiny_config
cfg=tiny_config();cfg.update(is_multimodal=False,out_audio_dim=4)
model=D.DiffusionTransformer3D(**cfg).eval();activate_gates(model)
video=torch.randn(1,2,4,4,4)
kw=dict(x_video=torch.cat((video,torch.zeros_like(video),torch.zeros_like(video[...,:1])),-1),
        x_audio=None,text_embed=torch.randn(1,5,12),pooled_text_embed=torch.randn(1,8),
        time=torch.tensor([500.]),visual_rope=model.visual_rope((2,2,2),[torch.arange(2)]*3),
        audio_rope=None,text_rope=model.text_rope(torch.arange(5)),
        attention_mask=torch.ones(1,5,dtype=torch.bool))
shapes={};hooks=[]
def capture(name):
    def hook(module,inputs,output):
        shapes[name]=list(output.shape)
    return hook
for name,mod in [('patches',model.visual_embeddings),('text_projection',model.text_embeddings),
                 ('text_refiner',model.text_blocks[0]),('visual_block',model.visual_transformer_blocks[0]),
                 ('out_layer',model.out_layer)]:
    hooks.append(mod.register_forward_hook(capture(name)))
with torch.no_grad():out=fresh_forward(model,kw)
for h in hooks:h.remove()
assert out.shape==video.shape
metrics['native_shapes']=shapes
print(json.dumps(shapes,indent=2))


## 3. Проверяем, что текст реально влияет через нужную ветвь

При zero-init gates влияние cross-attention может быть нулевым — это ожидаемая инициализация,
а не ошибка архитектуры. Мы уже включили небольшие случайные modulation weights для sensitivity probe.
Изменим Qwen-like token states, оставив CLIP-like pooled state неизменным.
Потом отключим только text cross-attention output через hook и повторим perturbation.
Так отделяется token-level путь от pooled/time modulation.


In [ ]:
kw2=dict(kw);kw2['text_embed']=kw['text_embed']+torch.randn_like(kw['text_embed'])
with torch.no_grad():
    base=fresh_forward(model,kw);perturbed=fresh_forward(model,kw2)
active_delta=(base-perturbed).abs().max().item()
handles=[b.cross_attention.register_forward_hook(lambda m,i,o:torch.zeros_like(o))
         for b in model.visual_transformer_blocks]
with torch.no_grad():
    cut_base=fresh_forward(model,kw);cut_perturbed=fresh_forward(model,kw2)
for h in handles:h.remove()
cut_delta=(cut_base-cut_perturbed).abs().max().item()
assert active_delta>0 and cut_delta<1e-6
metrics['token_condition_active_delta']=active_delta;metrics['token_condition_cut_delta']=cut_delta
print('Token perturbation: active=',active_delta,'text CA cut=',cut_delta)


## 4. Сопоставление архитектур по контрактам

| Архитектура | Обновляемые tokens | Где condition | Какое сравнение корректно |
|---|---|---|---|
| Original DiT | image latent patches | timestep/class conditioning, adaLN variants | patching, time conditioning, residual gating |
| Kandinsky text CrossDiT | video patches + отдельный text refiner | Qwen tokens → text cross-attention; pooled → time | native video-only forward выше |
| K6 dual-stream | video tokens и audio tokens разной width | два text-refiner + двунаправленная AV fusion | следующая лабораторная |
| SD3 MMDiT | modality-specific weights с совместным attention | совместное text/image token взаимодействие | сравнение дизайна, не буквальный K6 код |

[SD3 / Scaling Rectified Flow Transformers](https://arxiv.org/abs/2403.03206) полезен для сравнения
multimodal choices. Слово «multimodal» само по себе не доказывает одинаковый порядок операций.


## Самостоятельное исследование

1. Нарисуйте полный diagram shapes из hooks и подпишите, где число video tokens меняется.
2. При фиксированном token context измените pooled condition; какой путь останется даже после cut text CA?
3. Сравните 6 и 9 modulation vectors. В каких residual branches они используются?
4. Откройте DiT paper, K5 paper и native class: отметьте общие primitives и разные prediction heads.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы можете самостоятельно написать self-attention+AdaLN residual block и объяснить,
какие дополнительные модули превращают его в text-conditioned Kandinsky CrossDiT.
Архитектура блока не определяет ни solver, ни noise/velocity/grid target.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 06. Attention и RoPE: текст, видео и аудио](06_attention_and_rope.ipynb) · [08. Dual-stream CrossDiT K6: два потока Audio + Video →](08_dual_stream_audio_video_crossdit.ipynb)
